# PC2 — Grupo 4 · Programación Concurrente y Distribuida (1ACC0065)
## Parte 1: Modelado en Promela de la lógica de sincronización

**Proyecto:** Predicción concurrente del consumo energético urbano (Regresión Lineal, dataset *Smart Meters in London*, ODS 11).

Este cuaderno cubre el primer criterio de la PC2: *planteamiento del algoritmo concurrente inicial y modelado en Promela para simular la lógica de sincronización y demostrar que esté libre de condición de carrera* (3 puntos).

El cuaderno está organizado en cinco partes: (1) Promela, (2) ejecución de la regresión en Go del repositorio (`regresion/`), (3) speedup con media recortada, (4) recursos y guía de análisis y (5) evidencias para el repositorio.

| Paso | Contenido | Resultado esperado |
|---|---|---|
| 1 | Planteamiento del algoritmo concurrente | Diseño Worker Pool con acumulador protegido |
| 2 | Instalación de Spin | Spin 6.5.2 |
| 3 | Modelo sin protección (`race_condition.pml`) | Spin encuentra la condición de carrera |
| 4 | Contraejemplo paso a paso | Pérdida de actualización (*lost update*) |
| 5 | Modelo corregido (`race_condition_fixed.pml`) | 0 errores, 0 estados finales inválidos |
| 6 | Evidencias y equivalencia con Go | Archivos para el repositorio y el informe |

## 1. Planteamiento del algoritmo concurrente

El modelo es una **Regresión Lineal Múltiple por Ecuaciones Normales**:

$$\hat{\beta} = (X^\top X)^{-1} X^\top y$$

Como $X^\top X$ y $X^\top y$ son **sumas por filas**, el dataset de entrenamiento (≈2.56 millones de filas hogar-día, 11 variables más el intercepto) puede dividirse en bloques independientes:

$$X^\top X = \sum_{k=1}^{N} X_k^\top X_k \qquad\qquad X^\top y = \sum_{k=1}^{N} X_k^\top y_k$$

**Patrón elegido: Worker Pool.**

1. El productor divide las filas en *N* bloques y los envía por un canal de trabajos.
2. Cada *goroutine* worker calcula sus parciales $X_k^\top X_k$ y $X_k^\top y_k$ **en memoria local**, sin compartir nada.
3. Al terminar, el worker incorpora su parcial al **acumulador global** dentro de una **sección crítica** protegida por `sync.Mutex`.
4. `sync.WaitGroup` espera a todos los workers y luego se resuelve, de forma secuencial, el sistema 12×12.

El único punto con memoria compartida es el paso 3. Ese es el punto que se modela y verifica en Promela.

**Abstracción del modelo.** Promela no maneja matrices de números reales, así que el acumulador matricial se reduce a un entero: la suma parcial de `daily_energy_kwh` de cada bloque, que es un componente de $X^\top y$. Si la actualización de un solo componente es segura, la de toda la matriz también lo es, porque usa la misma sección crítica.

| Elemento en Go | Elemento en Promela |
|---|---|
| goroutine worker | `proctype Worker` |
| acumulador global `XtY` | `int global_sum_kwh` |
| parcial del bloque | parámetro `partial_kwh` (125 430, 98 650 y 143 210 kWh) |
| `sync.Mutex` `Lock()` / `Unlock()` | `atomic { !mutex -> mutex = true }` / `mutex = false` |
| `sync.WaitGroup` `Wait()` | `(finished == 3)` en `init` |
| verificación de resultado | `assert(global_sum_kwh == expected_sum)` |

## 2. Instalación de Spin

Colab usa Ubuntu, cuyo repositorio oficial incluye **Spin 6.5.2**, la misma versión con la que se obtuvieron las capturas del informe. También se usa `gcc`, que ya viene instalado, para compilar el verificador `pan`.

In [ ]:
!apt-get -qq install -y spin > /dev/null
!spin -V && gcc --version | head -1

## 3. Modelo sin protección: `race_condition.pml`

Tres workers suman su aporte al acumulador global en **tres pasos separados**: leer, sumar y escribir. Estos pasos no son atómicos, así que Spin puede intercalarlos de todas las formas posibles.

- Resultado correcto: 125 430 + 98 650 + 143 210 = **367 290 kWh**.
- La aserción final comprueba que el acumulador tenga ese valor cuando terminan los tres workers.

In [ ]:
%%writefile race_condition.pml
/* race_condition.pml
 * Modelo SIN protección: tres workers suman su aporte parcial (kWh)
 * a un acumulador global compartido. Se espera que Spin encuentre
 * una intercalación con pérdida de actualización (lost update).
 */
int  global_sum_kwh = 0;   /* acumulador global compartido          */
int  expected_sum   = 0;   /* suma correcta, calculada en init      */
byte finished       = 0;   /* contador de workers que terminaron    */

proctype Worker(int partial_kwh) {
    int local_sum;
    local_sum = global_sum_kwh;              /* 1. leer el acumulador   */
    local_sum = local_sum + partial_kwh;     /* 2. sumar el aporte      */
    global_sum_kwh = local_sum;              /* 3. escribir el resultado */
    finished++
}

init {
    expected_sum = 125430 + 98650 + 143210;  /* = 367290 kWh */
    run Worker(125430);
    run Worker(98650);
    run Worker(143210);
    (finished == 3);                         /* esperar a los 3 workers */
    assert(global_sum_kwh == expected_sum)
}

### 3.1 Verificación con Spin

Los tres comandos de siempre:

1. `spin -a` genera el verificador en C (`pan.c`).
2. `gcc` lo compila. La opción `-w` solo oculta advertencias del compilador que no afectan el resultado.
3. `./pan` explora el espacio de estados y guarda la salida en `resultados/` como evidencia.

In [ ]:
!mkdir -p resultados
!spin -a race_condition.pml && gcc -w -o pan pan.c && ./pan | tee resultados/race_condition_pan.txt

### 3.2 Interpretación

La salida debe coincidir con estos valores:

| Indicador | Valor | Significado |
|---|---|---|
| `assertion violated` | a profundidad 20 | Existe una ejecución que termina con una suma incorrecta |
| `errors` | 1 | Spin encontró la condición de carrera |
| `states stored / matched / transitions` | 39 / 3 / 42 | Tamaño de la búsqueda hasta hallar el error |
| `Search not completed` | — | Normal: Spin se detiene al primer error |

Spin también escribe `race_condition.pml.trail`, que contiene la ejecución exacta que produjo el error. La reproducimos en el paso siguiente.

## 4. Contraejemplo paso a paso

`spin -t -p -g -l` reproduce la traza e imprime cada paso con los valores de las variables globales (`-g`) y locales (`-l`). Si aparece el aviso *"race_condition.pml is newer than race_condition.pml.trail"*, puede ignorarse: se debe a que ambos archivos se crearon en el mismo segundo.

In [ ]:
!spin -t -p -g -l race_condition.pml | tee resultados/race_condition_trail.txt

### 4.1 Lectura de la traza

| Paso | Worker | Acción | Valor |
|---|---|---|---|
| 5–8 | Worker 3 (143 210) | lee 0, suma y escribe | `global_sum_kwh = 143 210` |
| 10–11 | Worker 2 (98 650) | lee 143 210 y suma en local | `local_sum = 241 860` |
| 12–13 | Worker 1 (125 430) | **lee el mismo 143 210** y suma en local | `local_sum = 268 640` |
| 14 | Worker 2 | escribe | `global_sum_kwh = 241 860` |
| 17 | Worker 1 | **sobrescribe** | `global_sum_kwh = 268 640` |
| 21 | init | `assert` falla | 268 640 ≠ 367 290 |

Los workers 1 y 2 leyeron el acumulador **antes** de que el otro escribiera. La segunda escritura borró a la primera y el aporte del worker 2 se perdió: 367 290 − 98 650 = 268 640. Esto es una **pérdida de actualización** (*lost update*), el caso típico de condición de carrera sobre memoria compartida.

## 5. Modelo corregido: `race_condition_fixed.pml`

Se añade un **mutex** que protege la secuencia leer-sumar-escribir:

- **Lock:** `atomic { !mutex -> mutex = true }`. El worker espera a que el mutex esté libre y lo toma en un solo paso indivisible. Sin `atomic`, dos workers podrían ver el mutex libre al mismo tiempo.
- **Sección crítica:** los tres pasos de actualización.
- **Unlock:** `mutex = false`.

El resto del modelo no cambia, para que la comparación entre ambas versiones sea directa.

In [ ]:
%%writefile race_condition_fixed.pml
/* race_condition_fixed.pml
 * Misma lógica, pero la sección crítica (leer-sumar-escribir)
 * está protegida por un mutex. Equivale a sync.Mutex en Go.
 */
int  global_sum_kwh = 0;
int  expected_sum   = 0;
byte finished       = 0;
bool mutex          = false;   /* false = libre, true = ocupado */

proctype Worker(int partial_kwh) {
    int local_sum;
    atomic { !mutex -> mutex = true };       /* Lock(): esperar y tomar  */
    local_sum = global_sum_kwh;              /* --- sección crítica ---  */
    local_sum = local_sum + partial_kwh;
    global_sum_kwh = local_sum;              /* ------------------------ */
    mutex = false;                           /* Unlock()                 */
    finished++
}

init {
    expected_sum = 125430 + 98650 + 143210;  /* = 367290 kWh */
    run Worker(125430);
    run Worker(98650);
    run Worker(143210);
    (finished == 3);
    assert(global_sum_kwh == expected_sum)
}

In [ ]:
!rm -f pan pan.* *.trail
!spin -a race_condition_fixed.pml && gcc -w -o pan pan.c && ./pan | tee resultados/race_condition_fixed_pan.txt

### 5.1 Interpretación

| Indicador | Valor | Significado |
|---|---|---|
| `errors` | **0** | Ninguna intercalación viola la aserción: no hay condición de carrera |
| `invalid end states` | + (activado), sin errores | Ningún proceso queda bloqueado: no hay *deadlock* |
| `states stored / matched / transitions` | 317 / 155 / 472 | Espacio de estados **completo** explorado |
| `depth reached` | 28 | Ejecución más larga posible |
| `unreached in proctype Worker` | 0 of 9 states | Se ejercitó todo el código del worker |
| `unreached in init` | 0 of 7 states | Se ejercitó todo el código de `init` |

A diferencia del caso anterior, aquí la búsqueda sí termina. Spin revisó **todas** las intercalaciones posibles de los tres workers y en todas el acumulador llegó a 367 290 kWh.

## 6. Evidencias y equivalencia con Go

### 6.1 Archivos para el repositorio

La celda siguiente reúne los modelos y las salidas de Spin en la carpeta `promela/`. Se descargan junto con el resto de evidencias al final del cuaderno (Parte 5).

In [ ]:
!mkdir -p promela && cp race_condition.pml race_condition_fixed.pml promela/ && cp -r resultados promela/
!ls -R promela

### 6.2 Cómo se traslada a Go

El diseño verificado corresponde a `BuildNormalEquationsConcurrent` en `regresion/normal_eq.go`. Se resume aquí como referencia (no es una celda ejecutable).

```go
var (
    mu  sync.Mutex
    wg  sync.WaitGroup
    XtX [P][P]float64   // acumulador global compartido
    XtY [P]float64
)

for w := 0; w < workers; w++ {
    wg.Add(1)
    go func() {
        defer wg.Done()
        for bloque := range jobs {
            pXtX, pXtY := parciales(bloque) // memoria local: sin sincronización
            mu.Lock()                       // Lock()  ≡ atomic { !mutex -> mutex = true }
            sumar(&XtX, &XtY, pXtX, pXtY)   // sección crítica ≡ leer-sumar-escribir
            mu.Unlock()                     // Unlock() ≡ mutex = false
        }
    }()
}
wg.Wait()                                   // ≡ (finished == 3)
beta := resolver(XtX, XtY)                  // secuencial, sistema 12×12
```

El mutex se toma **una sola vez por bloque**, no por cada fila. Así casi todo el cálculo ocurre en paralelo y la sección crítica es corta. Esto será relevante al analizar el speedup.

### 6.3 Checklist de capturas para el informe

1. Salida de la sección 3.1: `assertion violated`, `errors: 1`.
2. Traza de la sección 4, pasos 10 a 21.
3. Salida de la sección 5: `errors: 0`, `317 states`, `unreached 0 of 9 / 0 of 7`.

---
# Parte 2: Regresión en Go del repositorio (secuencial vs. concurrente)

Esta parte **no reescribe el código**: clona el repositorio del grupo y compila la carpeta `regresion/`, que es la misma implementación usada en el informe. Así el notebook, el informe y GitHub muestran un único código.

| Archivo | Responsabilidad |
|---|---|
| `data.go` | `LoadSamples`: lee `lcl_modelo_go.csv(.gz)` y filtra por `dataset_split` |
| `normal_eq.go` | `BuildNormalEquationsSequential`, `BuildNormalEquationsConcurrent` (goroutines + `sync.Mutex` + `sync.WaitGroup`) y `SolveLinearSystem` (Gauss con pivoteo) |
| `metrics.go` | `Evaluate`: RMSE y R² |
| `benchmark.go` | `RunBenchmark`: 20 repeticiones por configuración y media recortada |
| `resource_monitor.go` | `MeasureResources`: memoria (`runtime.MemStats`) y ciclos de GC |
| `main.go` | Modos `sequential`, `concurrent`, `benchmark` y `resources` |

**Modelo:** 11 variables explicativas + intercepto = **12 coeficientes** (sistema 12×12). Se entrena con 2 559 364 filas de *train* y se evalúa con 452 163 de *validation*.

## 2.1 Entorno de ejecución

El rendimiento depende del hardware. Colab gratuito suele asignar **2 vCPU**, por lo que aquí el speedup máximo posible es cercano a 2. Las cifras del informe se midieron en un i7-11800H (8 núcleos físicos, 16 lógicos).

In [ ]:
!lscpu | grep -E 'Model name|^CPU\(s\)|Thread\(s\) per core|Core\(s\) per socket'
!free -h | head -2

## 2.2 Instalación de Go

Versión oficial de Go; el programa solo usa la **biblioteca estándar**.

In [ ]:
!wget -q https://go.dev/dl/go1.22.12.linux-amd64.tar.gz -O /tmp/go.tgz && rm -rf /usr/local/go && tar -C /usr/local -xzf /tmp/go.tgz
import os
os.environ['PATH'] = '/usr/local/go/bin:' + os.environ['PATH']
!go version

## 2.3 Código del repositorio

Se clona la rama indicada en `RAMA`. Normalmente es `main`; si la unificación de `regresion/` aún no se integró, use `develop` o la rama del PR.

In [ ]:
RAMA = 'main'
REPO = 'https://github.com/MapleSyrup63/Programaci-nConcurrenteYDistribuida_TP_Grupo4.git'
!rm -rf repo && git clone -q --depth 1 -b {RAMA} {REPO} repo
!cd repo && git log --oneline -3 && ls regresion

## 2.4 Dataset

Se busca `lcl_modelo_go.csv.gz` en la carpeta de Drive usada en la PC1; si no está, se pide subirlo. Se copia al disco local de Colab porque leerlo desde Drive es lento.

In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')

DATA = Path('/content/drive/MyDrive/Grupo4_PCD_GO/lcl_modelo_go.csv.gz')
if not DATA.exists():
    from google.colab import files
    print('No se encontró en Drive. Suba lcl_modelo_go.csv.gz:')
    DATA = Path(next(iter(files.upload())))

!cp "{DATA}" /content/lcl_modelo_go.csv.gz
DATA = '/content/lcl_modelo_go.csv.gz'
!ls -lh {DATA}

## 2.5 Compilación y análisis estático

`gofmt` (formato) y `go vet` (errores comunes) deben terminar sin mensajes.

In [ ]:
!cd repo/regresion && gofmt -l . && go vet . && go build -o regresion . && echo "Compilación correcta"

## 2.6 Evidencia: ejecución secuencial y concurrente

In [ ]:
!mkdir -p resultados
!./repo/regresion/regresion -input {DATA} -mode sequential | tee resultados/ejecucion_secuencial.txt

In [ ]:
!./repo/regresion/regresion -input {DATA} -mode concurrent -workers 4 | tee resultados/ejecucion_concurrente.txt

### 2.6.1 ¿La concurrencia altera el modelo?

Se comparan los 12 coeficientes. Las diferencias deben ser del orden de $10^{-11}$ o menores y se deben al **orden de las sumas en punto flotante**, no a un error de sincronización. Una diferencia grande indicaría una condición de carrera.

In [ ]:
import re
def coef(path):
    return [float(x) for x in re.search(r'Coeficientes \(beta\): \[(.*?)\]', open(path).read()).group(1).split()]
s, c = coef('resultados/ejecucion_secuencial.txt'), coef('resultados/ejecucion_concurrente.txt')
print(f'{len(s)} coeficientes | diferencia máxima absoluta: {max(abs(a-b) for a, b in zip(s, c)):.2e}')

## 2.7 Detector de condiciones de carrera de Go

Promela verificó el **modelo**; `go run -race` verifica el **código real** e informa `WARNING: DATA RACE` si dos goroutines acceden a la misma variable sin sincronización. La ejecución instrumentada es más lenta (unos minutos, sobre todo por la carga del CSV).

In [ ]:
!cd repo/regresion && go run -race . -input {DATA} -mode concurrent -workers 8 2>&1 | tee ../../resultados/race_detector.txt | tail -4
!grep -q "DATA RACE" resultados/race_detector.txt && echo "ATENCIÓN: se detectaron condiciones de carrera" || echo "Resultado: 0 condiciones de carrera detectadas"

---
# Parte 3: Speedup y media recortada

**Protocolo** (implementado en `benchmark.go`):

1. Solo se mide la construcción de $X^\top X$ y $X^\top y$; la carga del CSV y la resolución 12×12 quedan fuera.
2. Configuraciones: secuencial y 1, 2, 4, 8, 16 y 32 workers, **20 repeticiones** cada una.
3. **Media recortada al 10 %**: de las 20 mediciones se descartan las 2 más bajas y las 2 más altas.

El programa escribe `benchmark_raw.csv` (cada medición) y `benchmark_summary.csv` (medias y speedup).

In [ ]:
!cd repo/regresion && ./regresion -input {DATA} -mode benchmark -workerlist 1,2,4,8,16,32 -repeats 20 -trim 0.1 | grep -E "Media recortada|Guardado"
!cp repo/regresion/benchmark_raw.csv repo/regresion/benchmark_summary.csv resultados/

## 3.1 Tabla estadística

Además de la media recortada se reporta la **dispersión**, que la rúbrica pide como estadística:

| Columna | Significado |
|---|---|
| `media_recortada_ms` | Media de las 16 mediciones centrales |
| `desv_est_ms`, `CV_%` | Dispersión; CV < 10 % indica mediciones estables |
| `speedup` | $S = T_{sec} / T_{conc}$ |
| `eficiencia` | $E = S / W$ |
| `karp_flatt` | Fracción serial experimental $e = \dfrac{1/S - 1/p}{1 - 1/p}$, con $p$ = núcleos efectivos |

Para analizar un CSV medido en otra PC (por ejemplo la de 8 núcleos), súbalo a `resultados/benchmark_raw.csv` y ajuste `NUCLEOS`.

In [ ]:
import pandas as pd, numpy as np, os
from scipy import stats

runs = pd.read_csv('resultados/benchmark_raw.csv')
NUCLEOS = os.cpu_count()   # ajustar si el CSV viene de otra máquina (p. ej. 8 o 16)

filas = []
for (modo, w), g in runs.groupby(['modo', 'workers']):
    t = g['tiempo_ms']
    filas.append({'modo': modo, 'workers': w, 'n': len(t),
                  'media_recortada_ms': stats.trim_mean(t, 0.10), 'mediana_ms': t.median(),
                  'desv_est_ms': t.std(ddof=1), 'CV_%': 100 * t.std(ddof=1) / t.mean(),
                  'min_ms': t.min(), 'max_ms': t.max()})
tabla = pd.DataFrame(filas).sort_values('workers').reset_index(drop=True)
t_seq = tabla.loc[tabla.modo == 'sequential', 'media_recortada_ms'].iloc[0]
tabla['speedup'] = t_seq / tabla['media_recortada_ms']
tabla['eficiencia'] = np.where(tabla.workers > 0, tabla['speedup'] / tabla['workers'].clip(lower=1), 1.0)
p = np.minimum(tabla['workers'], NUCLEOS)
tabla['karp_flatt'] = np.where(p > 1, (1 / tabla['speedup'] - 1 / p) / (1 - 1 / p), np.nan)
tabla.to_csv('resultados/tabla_speedup.csv', index=False)
print(f'Núcleos lógicos considerados: {NUCLEOS}')
tabla.round(3)

In [ ]:
import matplotlib.pyplot as plt

AZUL, GRIS, TINTA, TINTA2 = '#2a78d6', '#8f8e89', '#0b0b0b', '#52514e'
plt.rcParams.update({'font.size': 10, 'axes.edgecolor': '#c9c8c2', 'axes.labelcolor': TINTA2,
                     'xtick.color': TINTA2, 'ytick.color': TINTA2, 'axes.spines.top': False,
                     'axes.spines.right': False})
conc = tabla[tabla.modo == 'concurrent']
w = conc['workers'].to_numpy()

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(w, np.minimum(w, NUCLEOS), '--', color=GRIS, lw=1.2, label=f'Ideal limitado a {NUCLEOS} núcleos')
ax.axhline(1, color=GRIS, lw=0.8, ls=':')
ax.plot(w, conc['speedup'], '-o', color=AZUL, lw=2, ms=7, label='Speedup medido')
for x, s in zip(w, conc['speedup']):
    ax.annotate(f'{s:.2f}x', (x, s), textcoords='offset points', xytext=(0, 8), ha='center', color=TINTA, fontsize=9)
ax.set_ylim(0, max(conc['speedup'].max(), NUCLEOS) * 1.18)
ax.set_xscale('log', base=2); ax.set_xticks(w); ax.set_xticklabels(w)
ax.set_xlabel('Workers (goroutines)'); ax.set_ylabel('Speedup = T secuencial / T concurrente')
ax.set_title('Speedup vs. número de workers (media recortada 10 %)', color=TINTA, loc='left')
ax.grid(axis='y', color='#ecebe6'); ax.legend(frameon=False)
fig.tight_layout(); fig.savefig('resultados/speedup.png', dpi=200); plt.show()

fig, ax = plt.subplots(figsize=(7, 3.4))
ax.bar([str(x) for x in w], conc['eficiencia'], color=AZUL, width=0.55)
for i, e in enumerate(conc['eficiencia']):
    ax.annotate(f'{e:.2f}', (i, e), textcoords='offset points', xytext=(0, 4), ha='center', color=TINTA, fontsize=9)
ax.set_xlabel('Workers (goroutines)'); ax.set_ylabel('Eficiencia = Speedup / workers'); ax.set_ylim(0, 1.2)
ax.set_title('Eficiencia paralela', color=TINTA, loc='left'); ax.grid(axis='y', color='#ecebe6')
fig.tight_layout(); fig.savefig('resultados/eficiencia.png', dpi=200); plt.show()

fig, ax = plt.subplots(figsize=(7, 3.8))
datos = [runs[(runs.modo == r.modo) & (runs.workers == r.workers)]['tiempo_ms'] for r in tabla.itertuples()]
ax.boxplot(datos, patch_artist=True,
           boxprops=dict(facecolor='#cde2fb', color=AZUL), medianprops=dict(color=AZUL, lw=2),
           whiskerprops=dict(color=GRIS), capprops=dict(color=GRIS), flierprops=dict(markeredgecolor=GRIS))
ax.set_xticks(range(1, len(datos) + 1))
ax.set_xticklabels(['sec' if x == 0 else str(x) for x in tabla['workers']])
ax.set_xlabel('Configuración (sec = secuencial; número = workers)'); ax.set_ylabel('Tiempo XᵀX / Xᵀy (ms)')
ax.set_title('Dispersión de las mediciones por configuración', color=TINTA, loc='left')
ax.grid(axis='y', color='#ecebe6'); fig.tight_layout(); fig.savefig('resultados/dispersion.png', dpi=200); plt.show()

---
# Parte 4: Recursos y guía de análisis

## 4.1 Memoria y recolector de basura

`resource_monitor.go` ejecuta una vez cada configuración y registra `Alloc`, el incremento de `TotalAlloc`, `Sys` y los ciclos de GC.

In [ ]:
!cd repo/regresion && ./regresion -input {DATA} -mode resources -workerlist 1,2,4,8,16,32 | grep -E "modo=|Guardado"
!cp repo/regresion/resource_usage.csv resultados/
pd.read_csv('resultados/resource_usage.csv')

## 4.2 Preguntas que el informe debe responder con estos números

**1. ¿Qué pasa con 1 worker?** Secuencial y concurrente usan la misma función `accumulate`, así que con 1 worker el speedup debe ser **≈1.0x**: crear una goroutine y tomar el mutex una vez cuesta microsegundos. Si sale muy por debajo de 1, la causa está en la medición (por ejemplo, medir primero el secuencial sin calentamiento), no en la concurrencia.

**2. ¿Hasta dónde escala?** Crece casi linealmente hasta el número de **núcleos físicos**; los hilos lógicos aportan menos y, pasado el total de núcleos, la curva se aplana.

**3. ¿Qué limita el speedup?** La sección crítica se ejecuta **una vez por worker**, así que su contención es despreciable. Los límites reales son los núcleos disponibles y el **ancho de banda de memoria** (se recorren cientos de MB por ejecución). Una Karp–Flatt cercana a 0 confirma que la parte secuencial es mínima.

**4. Trade-offs.**

| Aspecto | Secuencial | Concurrente |
|---|---|---|
| Tiempo | Base | ≈ T / núcleos hasta saturar |
| Complejidad | Mínima | Goroutines, Mutex, WaitGroup y verificación (Promela, `-race`) |
| Memoria extra | 0 | Un acumulador 12×12 por worker (~1.2 KB), despreciable |
| Resultado numérico | Exacto | Diferencias de ~$10^{-11}$ por el orden de las sumas |
| Riesgo | Ninguno | Condición de carrera si se omite el Mutex (demostrado en Promela) |

**5. Punto de equilibrio.** Es la menor cantidad de workers a partir de la cual el speedup deja de crecer de forma apreciable mientras la eficiencia sigue cayendo; normalmente coincide con los núcleos físicos.

---
# Parte 5: Evidencias

Se empaquetan las salidas de Spin (`promela/resultados/`) y de Go (`resultados/`). Para el repositorio: las salidas de Spin van en `promela/resultados/`; los CSV y gráficos del benchmark, en `regresion/`. El dataset **no** se sube.

In [ ]:
!mkdir -p entrega && cp -r promela resultados entrega/
!cd entrega && zip -qr ../evidencias_pc2.zip . && cd .. && unzip -l evidencias_pc2.zip | tail -n +4 | head -30
from google.colab import files
files.download('evidencias_pc2.zip')